# **Taller 3: Diseño de Bloques Completos al Azar (DBCA) e Importación de Datos**

**Curso:** Diseño Experimental y Aplicaciones Estadísticas  
**Nivel:** Universitario (Pregrado)
**Instructor:** Experto en Diseño Experimental y Ciencia de Datos  

---

## 🚀 **Instrucciones para Google Colab (R Kernel)**

Si estás ejecutando este taller desde **Google Colab**, ten en cuenta lo siguiente:

1. **Entorno en la Nube:** Colab ejecuta el cuaderno en un servidor temporal. Cada vez que abras el cuaderno, necesitarás instalar los paquetes de R requeridos.
2. **Configuración Automática:** La celda de abajo instalará de forma silenciosa y automática todas las librerías necesarias (`ggplot2`, `performance`, `see`, `agricolae`, `DescTools`, `car`) sin inundar la pantalla con registros de descarga.
3. **Fácil Acceso a Datos:** El primer ejercicio descarga de forma remota el conjunto de datos desde GitHub, y el ejercicio evaluativo genera dinámicamente un archivo CSV regional en tu directorio temporal. ¡Todo es 100% interactivo y no requiere subir archivos manualmente!

*(Para ejecutar una celda, haz clic en el botón de reproducción ▶️ a la izquierda de la celda o presiona `Shift + Enter`)*

In [ ]:
# ==============================================================================
# 0. CONFIGURACIÓN DE ENTORNO EN GOOGLE COLAB / LOCAL (LIBRERÍAS)
# ==============================================================================

# Evitar diálogos interactivos de selección de espejo CRAN
options(repos = c(CRAN = "https://cloud.r-project.org"))

# Instalación y carga silenciosa de paquetes estadísticos
preparar_entorno <- function(paquetes) {
  for (p in paquetes) {
    if (!require(p, character.only = TRUE)) {
      message(paste("Instalando paquete:", p, "..."))
      install.packages(p, dependencies = TRUE, quiet = TRUE)
      library(p, character.only = TRUE)
    }
  }
  message("¡Entorno de R configurado con éxito!")
}

paquetes_requeridos <- c("ggplot2", "performance", "see", "agricolae", "DescTools", "car")
suppressMessages(preparar_entorno(paquetes_requeridos))


## 1. Fundamentos Teóricos del DBCA

### ¿Cuál es la diferencia crítica entre un DCA y un DBCA?
En el **Diseño Completamente al Azar (DCA)**, partimos del supuesto de que las condiciones ambientales o externas de las unidades experimentales son homogéneas (por ejemplo, macetas bajo condiciones controladas de invernadero). En este caso, cualquier variabilidad no atribuible al tratamiento se agrupa bajo el término del error aleatorio experimental.

Sin embargo, en condiciones reales de campo (agronomía, forestal, ganadería), la homogeneidad absoluta es casi imposible de lograr. Factores como gradientes de fertilidad del suelo, pendientes topográficas, incidencia de sombra, o diferencias de peso en animales, introducen variabilidad no deseada. Si usáramos un DCA en un terreno heterogéneo, esa variabilidad extra inflaría el **Error Residual ($SC_{\text{Error}}$)**, haciendo que el denominador del estadístico $F$ sea muy grande. Como consecuencia, disminuye el poder del test ANOVA, aumentando el riesgo de cometer un **Error Tipo II** (no detectar diferencias reales entre tratamientos).

El **Diseño de Bloques Completos al Azar (DBCA)** soluciona esto introduciendo una **restricción en la aleatorización**. Agrupamos las unidades experimentales heterogéneas en subgrupos llamados **Bloques** (o repeticiones). Cada bloque debe ser internamente homogéneo y contener todos los tratamientos (completitud). La asignación de tratamientos se realiza al azar *dentro* de cada bloque.

### El Principio de Bloqueo y el Error Residual
Al bloquear el experimento, la variabilidad del factor de bloqueo se extrae del error experimental residual:
$$SC_{\text{Error (DBCA)}} = SC_{\text{Total}} - SC_{\text{Tratamientos}} - SC_{\text{Bloques}}$$
De este modo, reducimos el Cuadrado Medio del Error ($CM_{\text{Error}}$), incrementando la precisión y la potencia del ANOVA global.

### Comparación Matemática de los Modelos Lineales

* **Modelo del DCA (Unidireccional):**
  $$Y_{ij} = \mu + \tau_i + \varepsilon_{ij}$$
  * Donde:
    * $Y_{ij}$: Variable de respuesta de la unidad experimental del tratamiento $i$ y repetición $j$.
    * $\mu$: Media general del experimento.
    * $\tau_i$: Efecto del tratamiento $i$.
    * $\varepsilon_{ij}$: Error experimental aleatorio $\varepsilon_{ij} \sim N(0, \sigma^2)$ independientes.

* **Modelo del DBCA (Aditivo Lineal):**
  $$Y_{ij} = \mu + \tau_i + \beta_j + \varepsilon_{ij}$$
  * Donde se añade el término:
    * $\beta_j$: **Efecto de Bloque $j$**. Este término representa la desviación sistemática atribuible a la ubicación, gradiente o bloque experimental $j$. Se asume que no existe interacción entre los tratamientos y los bloques (modelo aditivo lineal).

---

### La Pérdida de Grados de Libertad: El "Costo" del Bloqueo
El bloqueo no es gratuito. Introducir bloques consume **Grados de Libertad (GL)** que de otro modo pertenecerían al término del error residual. Veamos la comparación matemática de la distribución de los Grados de Libertad:

| Fuente de Variación | DCA (GL) | DBCA (GL) | Ejemplo Numérico ($t=3, b=3$) |
| :--- | :--- | :--- | :--- |
| **Tratamientos (t)** | $t - 1$ | $t - 1$ | $2$ |
| **Bloques (b)** | — | $b - 1$ | $2$ |
| **Error Residual** | $N - t = t(b - 1)$ | $(t - 1)(b - 1)$ | DCA: $6$ GL vs. DBCA: $4$ GL |
| **Total (N - 1)** | $N - 1 = tb - 1$ | $tb - 1$ | $8$ |

**¿Por qué es esto importante en agronomía?**
* **En el DCA:** Los grados de libertad del error son mayores ($N - t$). Con más grados de libertad en el error, el estimador del Cuadrado Medio del Error ($CM_{\text{Error}}$) es más preciso y el valor crítico de la distribución $F$ ($F_{\text{crítico}}$) es más bajo, facilitando alcanzar la significancia estadística.
* **En el DBCA:** "Gastamos" $b - 1$ grados de libertad en el factor Bloque. Esto reduce los grados de libertad del error a $(t - 1)(b - 1)$.
* **El Dilema del Bloqueo:** Si el factor de bloqueo **no es efectivo** (es decir, el terreno era homogéneo y no había un gradiente real), habremos perdido grados de libertad en el error sin reducir la Suma de Cuadrados del Error ($SC_{\text{Error}}$). Esto provocará que el $CM_{\text{Error}} = SC_{\text{Error}} / GL_{\text{Error}}$ aumente, disminuyendo el estadístico $F$ calculado y restando **potencia estadística** al diseño.
* **Regla de Oro:** Solo se debe bloquear cuando exista una sospecha fundamentada o evidencia de heterogéneos gradientes ambientales en las unidades experimentales.

---

### Identificación de la Unidad Experimental
En diseño experimental, la **Unidad Experimental** es la fracción mínima de material experimental a la que se le aplica de forma independiente y aleatoria un tratamiento.
* **En nuestro ejercicio principal (Papa):** La unidad experimental **NO** es una planta individual de papa, ni tampoco es el bloque entero. La unidad experimental es **la parcela individual de campo (sub-parcela)** (por ejemplo, un área delimitada de 4m x 5m con varias plantas) donde se siembra una variedad específica de papa dentro de un bloque determinado. El rendimiento se mide por parcela y luego se reporta en toneladas por hectárea ($t/ha$).

---

### Esquema de Bloqueo en el Campo (Croquis de Distribución)
Supongamos que nuestro campo experimental de papas tiene un gradiente de humedad en el suelo que corre de norte a sur (Norte es más seco, Sur es más húmedo debido a acumulación freática).

Para controlar este gradiente, los bloques se establecen **perpendiculares al gradiente** (es decir, franjas horizontales de este a oeste), garantizando que dentro de cada bloque las condiciones de humedad sean lo más homogéneas posible.

A continuación se muestra el croquis de campo con la aleatorización de las 3 Variedades de Papa ($V_1$, $V_2$, $V_3$) en los 3 Bloques ($B_1$, $B_2$, $B_3$):

```
                   [ DIRECCIÓN DEL GRADIENTE DE HUMEDAD: SECO ]
Norte  ========================================================================
       BLOQUE 1 (Zona Seca - Terreno homogéneamente alto)
       +-----------------------+-----------------------+-----------------------+
       |   Variedad 2 (V2)     |   Variedad 1 (V1)     |   Variedad 3 (V3)     |  <- Aleatorio
       +-----------------------+-----------------------+-----------------------+
       ========================================================================
       BLOQUE 2 (Zona de Humedad Intermedia)
       +-----------------------+-----------------------+-----------------------+
       |   Variedad 1 (V1)     |   Variedad 3 (V3)     |   Variedad 2 (V2)     |  <- Aleatorio
       +-----------------------+-----------------------+-----------------------+
       ========================================================================
       BLOQUE 3 (Zona Húmeda - Terreno homogéneamente bajo)
       +-----------------------+-----------------------+-----------------------+
       |   Variedad 3 (V3)     |   Variedad 2 (V2)     |   Variedad 1 (V1)     |  <- Aleatorio
       +-----------------------+-----------------------+-----------------------+
Sur    ========================================================================
                   [ DIRECCIÓN DEL GRADIENTE DE HUMEDAD: HÚMEDO ]
```

* **Nota Crítica de Aleatorización:** Observa cómo las tres variedades aparecen exactamente una vez dentro de cada bloque, pero su orden de aparición espacial es completamente al azar en cada uno. Esto cumple con la **restricción de aleatorización** característica del DBCA.

## 2. Configuración del Entorno de R

Para realizar análisis de bloques, visualización avanzada y validación de supuestos, utilizaremos una suite de paquetes especializados de R. 

*(Nota: Si estás usando **Google Colab**, la celda autoloader del inicio ya instaló y cargó todo de forma silenciosa. Si trabajas de forma **local**, puedes ejecutar la celda de abajo para configurar tu entorno sin interrupciones interactivas)*

In [ ]:
# Evitar diálogos interactivos de CRAN
options(repos = c(CRAN = "https://cloud.r-project.org"))

# Carga/Instalación no interactiva de librerías
paquetes <- c("ggplot2", "performance", "see", "agricolae", "DescTools", "car")
for (p in paquetes) {
  if (!require(p, character.only = TRUE)) {
    install.packages(p, dependencies = TRUE, quiet = TRUE)
    library(p, character.only = TRUE)
  }
}
message("¡Entorno de R listo!")

## 3. Importación de Datos y Estructuración en R

### Importación desde Repositorios Públicos
En la ciencia de datos moderna y el diseño experimental, la reproducibilidad exige trabajar con datos alojados de forma remota (como en repositorios de GitHub). Esto evita depender de archivos locales que puedan perderse o estar mal direccionados.

### El Formato Regional e Importación con `read.csv2()`
En Latinoamérica y Europa, la configuración regional por defecto en hojas de cálculo (Excel) suele exportar archivos CSV con **punto y coma (;) como separador de columnas** y la **coma (,) como indicador de decimales** (por ejemplo, `3,14` en lugar de `3.14`). En R:
* La función estándar `read.csv()` asume comas (`,`) como separador y punto (`.`) como decimal.
* Si importamos un archivo regional latinoamericano con `read.csv()`, R fallará o agrupará toda la fila en una sola celda de texto.
* Por ello, es mandatorio utilizar **`read.csv2()`**, la cual está pre-configurada exactamente para reconocer el separador de punto y coma (`;`) y el punto decimal en coma (`,`).

### La Conversión Mandatoria a Factores (`as.factor()`)

**¿Por qué es obligatorio convertir las variables de agrupación a factores antes del ANOVA?**
Si las variables como `Variedad` y `Bloque` se dejan en formato numérico o de caracteres sin convertir, R interpretará en el ANOVA que son variables numéricas continuas y ajustará una **regresión lineal simple** (asignándole un solo grado de libertad), en lugar de tratarlas como niveles discretos de un factor. Para realizar un análisis de varianza correcto, es indispensable forzar su estructura al tipo categórico usando la función **`as.factor()`**.

In [ ]:
# Dirección web del archivo csv en GitHub
url_papa <- 'https://raw.githubusercontent.com/PALP31/BioAgro-Stats/main/01_Exploracion_Supuestos/datos_papa_campo.csv'

# Importación adecuada para formato regional latinoamericano
datos_papa <- read.csv2(url_papa)

# Conversión obligatoria de los factores de tratamiento y bloqueo
datos_papa$Variedad <- as.factor(datos_papa$Variedad)
datos_papa$Bloque   <- as.factor(datos_papa$Bloque)

# Verificación de los datos y su estructura interna
head(datos_papa)
str(datos_papa)

## 4. Análisis Gráfico de Bloques y Aditividad

### El Concepto de Aditividad
El modelo matemático del DBCA asume **aditividad**. Esto significa que los efectos de los tratamientos (Variedades) y los efectos de los Bloques se suman de manera independiente sin interactuar entre sí. En la práctica agrícola, esto implica que si la variedad A rinde más que la variedad B en un bloque fértil, también debería mantener ese comportamiento superior en un bloque menos fértil.

### Gráficos de Interacción o Perfiles Promedio (`interaction.plot()`)
El gráfico de perfiles es la mejor herramienta visual para comprobar la aditividad:
* Si las líneas de los tratamientos a través de los bloques corren de forma **paralela o similar** (sin cruzarse drásticamente), se confirma que se cumple la **aditividad**.
* Si las líneas se **cruzan drásticamente**, indica la presencia de una **interacción tratamiento-bloque**, lo cual violaría los supuestos del DBCA y requeriría técnicas de análisis avanzadas.

In [ ]:
interaction.plot(
  x.factor = datos_papa$Bloque,
  trace.factor = datos_papa$Variedad,
  response = datos_papa$Rendimiento,
  type = 'b',
  pch = c(19, 17, 15),
  col = c('blue', 'red', 'darkgreen'),
  xlab = 'Bloques (Gradiente de Humedad)',
  ylab = 'Rendimiento Promedio (t/ha)',
  main = 'Gráfico de Perfil: Rendimiento de Variedades de Papa'
)

## 5. Ajuste del Modelo e Interpretación del ANOVA

### La Fórmula Aditiva en R
En R, expresamos la estructura aditiva del DBCA usando el operador **`+`**. La fórmula `Rendimiento ~ Variedad + Bloque` le indica a la función `aov()` que evalúe simultáneamente el efecto de la `Variedad` (el tratamiento principal) y el efecto del `Bloque` (la variable de control ambiental), estimando los grados de libertad y sumas de cuadrados por separado para cada factor.

In [ ]:
# Ajuste del modelo aditivo para DBCA
modelo_papa <- aov(Rendimiento ~ Variedad + Bloque, data = datos_papa)

# Visualización de la tabla del Análisis de Varianza (ANOVA)
summary(modelo_papa)

## 6. Validación de los Supuestos del Modelo (Residuos)

Antes de interpretar el ANOVA para tomar decisiones comerciales, debemos validar que los residuos del modelo lineal ajustado cumplan estrictamente con los supuestos teóricos:

1. **Distribución general (Diagnóstico visual):** Evaluamos visualmente los residuos usando el paquete `performance` para asegurar la sanidad de los supuestos.
2. **Normalidad (Shapiro-Wilk):** Evaluamos cuantitativamente la distribución normal de los residuos ($H_0$: Residuos normales).
3. **Homocedasticidad (Prueba de Levene):** Evaluamos si existe homogeneidad de varianza entre tratamientos utilizando la prueba de Levene ($H_0$: Varianzas iguales).

In [ ]:
# 6.1 Diagnóstico gráfico de residuos automatizado con performance
check_model(modelo_papa, check = c('normality', 'homogeneity'))

In [ ]:
# 6.2 Prueba Formal Cuantitativa de Normalidad: Shapiro-Wilk (H0: Residuos normales)
residuos_papa <- residuals(modelo_papa)
shapiro.test(residuos_papa)

In [ ]:
# 6.3 Prueba Formal Cuantitativa de Homocedasticidad: Levene (H0: Varianzas iguales)
# Usamos la función LeveneTest de la librería DescTools
LeveneTest(Rendimiento ~ Variedad, data = datos_papa)

## 7. Pruebas de Comparaciones Múltiples de Medias (Post-Hoc)

Una vez verificado que se cumplen los supuestos, si el ANOVA global resulta significativo para el factor `Variedad` ($p < 0.05$), realizamos la prueba de **Tukey HSD** (Diferencia Significativa Honesta) utilizando la librería `agricolae`. Esta prueba controla estrictamente el error de tipo I (falsos positivos) al hacer todas las comparaciones de medias simultáneas.

In [ ]:
# Comparación múltiple de medias con Tukey HSD (agricolae)
tukey_papa <- HSD.test(modelo_papa, "Variedad", group = TRUE, console = TRUE)

## 8. Visualización Avanzada de Comparaciones de Medias y Letras

La forma ideal de comunicar resultados científicos es mostrar visualmente las diferencias significativas de las medias junto a sus **barras de error estándar** e imprimir las **letras de grupos de Tukey** directamente sobre ellas. Esto ayuda a comprender rápidamente qué variedades se solapan estadísticamente y cuáles destacan.

Dividimos esta visualización en dos celdas:
1. **Gráfica Nivel Básico (Nativa):** Utilizando el método `plot()` integrado de la librería `agricolae`.
2. **Gráfica Nivel Premium (ggplot2):** Construyendo un diseño gráfico de alta calidad editorial con barras de error estándar de la media (EEM) y letras de Tukey.

In [ ]:
# 8.1 Gráfica rápida nativa de agricolae
plot(tukey_papa, variation = 'SE', col = 'skyblue', main = 'Comparación de Variedades de Papa')

In [ ]:
# 8.2 Visualización Premium con ggplot2, Medias, EEM y Letras de Tukey
# Extracción del dataframe con los resultados de las medias y su error estándar
df_tukey_papa <- data.frame(
  Variedad = rownames(tukey_papa$means),
  Media = tukey_papa$means$Rendimiento,
  SD = tukey_papa$means$std,
  Rep = tukey_papa$means$r
)

# Error Estándar de la Media (EEM)
df_tukey_papa$EEM <- df_tukey_papa$SD / sqrt(df_tukey_papa$Rep)
df_tukey_papa$Grupo <- tukey_papa$groups[rownames(df_tukey_papa), 'groups']

# Gráfico premium con ggplot2
ggplot(df_tukey_papa, aes(x = Variedad, y = Media, fill = Variedad)) +
  geom_bar(stat = 'identity', color = 'black', alpha = 0.7, width = 0.5) +
  geom_errorbar(aes(ymin = Media - EEM, ymax = Media + EEM), width = 0.15, size = 0.8) +
  geom_text(aes(label = Grupo, y = Media + EEM + 0.4), size = 6, fontface = 'bold') +
  scale_fill_brewer(palette = 'Set1') +
  theme_minimal(base_size = 14) +
  labs(
    title = 'Rendimiento Promedio de Variedades de Papa en DBCA',
    subtitle = 'Letras distintas indican diferencias significativas (Tukey p < 0.05). Barras representan el EEM.',
    x = 'Variedad de Papa',
    y = 'Rendimiento Promedio (t/ha)'
  ) +
  theme(legend.position = 'none')

## 9. Cuestionario de Análisis del Experimento

Responde de manera detallada y con fundamento crítico cada una de las siguientes interrogantes basándote en los análisis estadísticos anteriores:

1. **¿El factor de Bloqueo (gradiente de humedad) resultó ser estadísticamente significativo en el ANOVA? ¿Qué nos dice esto sobre la decisión de haber "bloqueado" el terreno en lugar de usar un DCA simple?**
   
   *Escribe tu respuesta aquí:*
   
2. **¿Cuáles variedades de papa presentaron los rendimientos promedio más altos y más bajos? ¿Difieren estadísticamente entre sí según la prueba de Tukey?**
   
   *Escribe tu respuesta aquí:*
   
3. **Al observar el Gráfico de Perfil (`interaction.plot`), ¿las líneas se cruzan de forma drástica o mantienen un comportamiento mayoritariamente aditivo? ¿Qué implicancia tiene esto en el cumplimiento de los supuestos del DBCA?**
   
   *Escribe tu respuesta aquí:*
   
4. **Si tuvieras que recomendarle una variedad de papa a un productor que tiene problemas de terrenos con humedad variable, ¿cuál recomendarías basándote estrictamente en tus resultados estadísticos y por qué?**
   
   *Escribe tu respuesta aquí:*

# **ACTIVIDAD PRÁCTICA AUTÓNOMA (Taller de Aplicación)**

**Contexto Agrícola:**  
Una cooperativa de agricultores ecológicos en los Andes evalúa el rendimiento de papa bajo **4 tipos de abonos orgánicos** (`Control` [sin abono], `Compost`, `Humus de lombriz` y `Guano de Isla`). 

Debido a que el terreno experimental cuenta con una **pendiente topográfica pronunciada**, la cual genera variaciones sistemáticas de escorrentía, lavado de nutrientes y acumulación de humedad desde la parte alta a la baja, los investigadores dividieron el terreno en **5 bloques (altitudinales)** para controlar este gradiente de humedad.

**Identificación del Diseño y Conceptos Clave (Preguntas de preparación):**
*   **¿Cuál es la Unidad Experimental en este ejercicio?**  
    La unidad experimental es la sub-parcela individual de terreno (por ejemplo, una sección delimitada de la ladera con varias plantas de papa) que recibe de forma aleatoria e independiente uno de los tratamientos de abono orgánico dentro de un bloque específico. 
*   **¿Cómo se disponen los bloques en el campo?**  
    Los bloques de parcelas se disponen **perpendiculares al gradiente de la pendiente** (en franjas horizontales que sigan la misma línea altitudinal), para asegurar la máxima homogeneidad dentro de cada bloque, mientras que el gradiente sistemático corre verticalmente de arriba a abajo.

**El Reto de la Actividad:**  
Para simular un escenario profesional de consultoría en ciencia de datos agrícolas, el docente ha exportado físicamente un archivo CSV regional llamado `tarea_dbca_abonos.csv` en tu espacio de trabajo. Tu labor es realizar la importación de este archivo de datos y conducirlo a través del flujo analítico del DBCA de manera rigurosa, resolviendo cada paso por separado.

In [ ]:
# ==============================================================================
# DOCENTE: GENERACIÓN FÍSICA DEL ARCHIVO CSV DE LA TAREA
# ==============================================================================

# Creamos el data.frame con los datos del ensayo agrícola
datos_abonos <- data.frame(
  Bloque = factor(rep(c("Bloque_1", "Bloque_2", "Bloque_3", "Bloque_4", "Bloque_5"), each = 4)),
  Abono = factor(rep(c("Control", "Compost", "Humus", "Guano"), times = 5)),
  Rendimiento_tha = c(
    13.2, 17.5, 20.3, 23.8,  # Bloque 1 (Alta pendiente, menos fértil)
    12.9, 16.9, 19.8, 23.1,  # Bloque 2
    14.1, 18.4, 21.4, 24.9,  # Bloque 3
    13.5, 17.2, 20.0, 23.5,  # Bloque 4
    14.8, 18.9, 22.0, 25.7   # Bloque 5 (Baja pendiente, más acumulación)
  )
)

# Escribimos físicamente el archivo CSV utilizando configuración regional Latinoamericana
# separador punto y coma (;) y coma para los decimales (,) mediante write.csv2
write.csv2(datos_abonos, file = "tarea_dbca_abonos.csv", row.names = FALSE)

print("¡El archivo CSV regional 'tarea_dbca_abonos.csv' ha sido creado físicamente en el directorio!")
head(datos_abonos)

### **Paso 1: Importación del CSV Regional**

El archivo `tarea_dbca_abonos.csv` se ha guardado en el disco con formato regional latinoamericano (delimitador punto y coma `;` e indicador decimal coma `,`).

**Instrucciones:** Escribe la sentencia de R adecuada para importar este archivo correctamente. Guarda el resultado en una variable llamada `datos_tarea` y visualiza la estructura usando `str()` y las primeras líneas con `head()`.

In [ ]:
# [Escribe tu código para importar el archivo CSV regional aquí]


### **Paso 2: Conversión a Factores**

**Instrucciones:** Asegura que las columnas de agrupación `Abono` (tratamiento) y `Bloque` (bloqueo) sean interpretadas correctamente por R como categorías. Aplica `as.factor()` sobre ambas columnas en el dataframe `datos_tarea` y comprueba el cambio con `str()`.

In [ ]:
# [Escribe tu código para la conversión obligatoria a factores aquí]


### **Paso 3: Análisis Gráfico de Perfiles e Interacción**

**Instrucciones:** Construye el gráfico de interacción (`interaction.plot`) colocando en el eje X el factor de bloqueo `Bloque`, el factor de tratamiento `Abono` en la leyenda trazadora, y el rendimiento `Rendimiento_tha` como respuesta. Evalúa si se cumple el supuesto de aditividad.

In [ ]:
# [Escribe tu código para construir el gráfico de perfiles de interacción aquí]


### **Paso 4: Ajuste del Modelo DBCA y Tabla ANOVA**

**Instrucciones:** Ajusta el modelo lineal aditivo para el diseño de bloques completos al azar (incluyendo el factor `Abono` y el factor `Bloque`). Asigna el modelo ajustado a la variable `modelo_tarea` y despliega la tabla ANOVA global usando `summary()`.

In [ ]:
# [Escribe tu código para ajustar el modelo y desplegar la tabla ANOVA aquí]


### **Paso 5: Validación de los Supuestos del Modelo (Residuos)**

Para asegurar que tus inferencias científicas y p-valores sean válidos, valida de forma separada los supuestos teóricos del modelo sobre sus residuos:

#### **5.1 Diagnóstico Gráfico Integral**
**Instrucciones:** Utiliza `check_model()` de la librería `performance` para inspeccionar visualmente la distribución de la normalidad y la homogeneidad de varianza.

In [ ]:
# [Escribe tu código para el diagnóstico gráfico automatizado de residuos aquí]


#### **5.2 Prueba Formal de Normalidad**
**Instrucciones:** Extrae los residuos de `modelo_tarea` y evalúa cuantitativamente la normalidad mediante la prueba de `shapiro.test()`.

In [ ]:
# [Escribe tu código para evaluar la normalidad mediante Shapiro-Wilk aquí]


#### **5.3 Prueba Formal de Homocedasticidad**
**Instrucciones:** Evalúa si las varianzas son homogéneas entre los diferentes abonos utilizando la función `LeveneTest()` de la librería `DescTools`.

In [ ]:
# [Escribe tu código para evaluar la homocedasticidad mediante LeveneTest aquí]


### **Paso 6: Comparación de Medias mediante Tukey HSD**

**Instrucciones:** Si el análisis de varianza indicó un efecto significativo del factor abono, ejecuta la prueba post-hoc de Tukey HSD utilizando `HSD.test()` de la librería `agricolae` sobre `modelo_tarea` para agrupar las medias de los abonos. Imprime la tabla resultante en la consola y guárdala en la variable `tukey_tarea`.

In [ ]:
# [Escribe tu código para ejecutar el test Tukey HSD aquí]


### **Paso 7: Visualización Avanzada de Medias y Letras de Tukey**

Comunica los resultados de manera visual a los agricultores de la cooperativa:

#### **7.1 Gráfica Básica (Nativa)**
**Instrucciones:** Genera la gráfica rápida nativa aplicando la función `plot()` sobre el objeto `tukey_tarea`.

In [ ]:
# [Escribe tu código para generar la gráfica nativa de Tukey aquí]


#### **7.2 Gráfica Premium con ggplot2**
**Instrucciones:** Extrae los promedios y errores estándar desde `tukey_tarea` y construye un gráfico de barras premium con ggplot2. Asegura que muestre las barras de medias, las barras de error estándar de la media (EEM) y las correspondientes letras de agrupación del test de Tukey sobre cada abono.

In [ ]:
# [Escribe tu código para elaborar la visualización premium de ggplot2 aquí]


### **Paso 8: Cuestionario Final de Conclusiones Técnicas**

Responde de forma crítica y con rigor científico las siguientes preguntas basadas en tu análisis:

1. **¿Qué tipo de abono orgánico recomendarías a la cooperativa para maximizar el rendimiento del cultivo de papa y por qué? Fundamenta tu respuesta utilizando las letras de agrupación del test de Tukey.**
   
   *Escribe tu respuesta aquí:*
   
2. **¿Qué nos indica la significancia del factor 'Bloque' en el ANOVA de tu tarea? ¿Habría sido adecuado realizar este experimento en un terreno con pendiente usando un diseño completamente al azar (DCA) tradicional?**
   
   *Escribe tu respuesta aquí:*
   
3. **¿Se cumplieron de manera satisfactoria los supuestos de Normalidad y Homocedasticidad en los residuos? Justifica tu respuesta citando detalladamente los valores $p$ obtenidos en las pruebas de Shapiro-Wilk y Levene.**
   
   *Escribe tu respuesta aquí:*
   
4. **Basándote en el gráfico de interacción de perfiles promedios, ¿los efectos del bloque y del tratamiento se comportan de manera aditiva? Justifica gráficamente.**
   
   *Escribe tu respuesta aquí:*